# 02 — Selective Scan in PyTorch

This notebook walks through the selective scan mechanism — the core computation in Mamba.

**What you will learn:**
1. How selective scan differs from classical SSM recurrence
2. The role of each parameter ($u$, $\Delta$, $A$, $B$, $C$, $D$, $z$)
3. Shape contracts and the recurrence loop
4. Validating our implementation against the official `mamba-130m-hf` model


## From Classical SSM to Selective Scan

In notebook 01, we had fixed $A$, $B$, $C$. Mamba makes $B_t$, $C_t$, and $\Delta_t$ **input-dependent**:

| Parameter | Classical SSM | Mamba Selective Scan |
|-----------|--------------|---------------------|
| $A$ | Fixed matrix | Fixed (diagonal, log-parameterized) |
| $B$ | Fixed matrix | $B_t = \text{Linear}(x_t)$ — per-token |
| $C$ | Fixed matrix | $C_t = \text{Linear}(x_t)$ — per-token |
| $\Delta$ | Fixed scalar | $\Delta_t = \text{softplus}(\text{Linear}(x_t))$ — per-token |

The recurrence becomes:
$$\bar{A}_t = e^{\Delta_t \cdot A}$$
$$x_t = \bar{A}_t \odot x_{t-1} + \Delta_t \cdot B_t \cdot u_t$$
$$y_t = C_t^\top x_t + D \cdot u_t$$

**Why selectivity matters:** By making $\Delta$ input-dependent, the model can control its effective "gate" — large $\Delta$ means "listen to the input", small $\Delta$ means "keep the old state".


## Shape Contracts

All tensors follow these conventions (from `CONTRACTS.md`):

| Tensor | Shape | Description |
|--------|-------|-------------|
| `u` | `(B, D, L)` | Input sequence |
| `delta` | `(B, D, L)` | Per-token time steps (pre-softplus) |
| `A` | `(D, N)` | Continuous transition (negative, log-parameterized) |
| `B` | `(B, N, L)` or `(B, D, N, L)` | Input projection |
| `C` | `(B, N, L)` or `(B, D, N, L)` | Output projection |
| `D` | `(D,)` | Skip connection |
| `z` | `(B, D, L)` | SiLU gate |
| `y` | `(B, D, L)` | Output |


In [ ]:
import torch
from mamba_minimal.selective_scan import selective_scan_ref

torch.manual_seed(42)

# Create synthetic inputs matching the shape contracts
B, D, N, L = 2, 8, 4, 16

u = torch.randn(B, D, L)
delta = torch.rand(B, D, L)           # will be softplus'd inside
A = -torch.rand(D, N)                 # negative for stability
B_t = torch.randn(B, N, L)            # shared B across channels
C_t = torch.randn(B, N, L)            # shared C across channels
D_skip = torch.randn(D)
z = torch.randn(B, D, L)

# Run the reference selective scan
y = selective_scan_ref(u, delta, A, B_t, C_t, D=D_skip, z=z)

print(f"Input shape:  u = {tuple(u.shape)}")
print(f"Output shape: y = {tuple(y.shape)}")
print(f"Output range: [{y.min().item():.4f}, {y.max().item():.4f}]")
print(f"Output is finite: {torch.isfinite(y).all().item()}")

## Step-by-Step Recurrence Visualization

Let us trace through the recurrence for a single batch element and channel to see how the state evolves:


In [ ]:
import matplotlib.pyplot as plt
import torch.nn.functional as F

# Small example we can visualize
B_viz, D_viz, N_viz, L_viz = 1, 1, 4, 32
torch.manual_seed(0)

u_viz = torch.sin(torch.linspace(0, 6, L_viz)).unsqueeze(0).unsqueeze(0)  # (1, 1, 32)
delta_viz = torch.full((1, 1, L_viz), 0.5)
A_viz = -torch.tensor([[0.5, 1.0, 2.0, 4.0]])  # (1, 4) — different decay rates
B_viz_t = torch.ones(1, N_viz, L_viz) * 0.3
C_viz_t = torch.ones(1, N_viz, L_viz)

# Run scan and also get the state trajectory
delta_pos = F.softplus(delta_viz)
delta_A = torch.exp(torch.einsum("bdl,dn->bdln", delta_pos, A_viz))
delta_B_u = torch.einsum("bdl,bnl,bdl->bdln", delta_pos, B_viz_t, u_viz)

x = torch.zeros(1, 1, N_viz)
states = []
outputs = []
for t in range(L_viz):
    x = delta_A[:, :, t, :] * x + delta_B_u[:, :, t, :]
    y_t = torch.einsum("bdn,bn->bd", x, C_viz_t[:, :, t])
    states.append(x[0, 0].clone())
    outputs.append(y_t[0, 0].item())

states = torch.stack(states)

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(u_viz[0, 0].numpy(), "k-", linewidth=2, label="input u")
axes[0].set_ylabel("Input")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for n in range(N_viz):
    axes[1].plot(states[:, n].numpy(), label=f"state dim {n} (A={A_viz[0,n].item():.1f})")
axes[1].set_ylabel("Hidden state")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

axes[2].plot(outputs, "r-", linewidth=2, label="output y")
axes[2].set_ylabel("Output")
axes[2].set_xlabel("Time step")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

fig.suptitle("Selective Scan Recurrence: State Evolution", fontsize=13)
fig.tight_layout()
plt.show()

print("Notice: larger |A| values cause faster decay (state dim 3 forgets quickly)")

## Gradient Flow

The selective scan is differentiable. Let us verify gradients flow through the recurrence:


In [ ]:
torch.manual_seed(0)
B_g, D_g, N_g, L_g = 2, 4, 3, 8

u_g = torch.randn(B_g, D_g, L_g, requires_grad=True)
delta_g = torch.rand(B_g, D_g, L_g, requires_grad=True)
A_g = -torch.rand(D_g, N_g, requires_grad=True)
B_g_t = torch.randn(B_g, N_g, L_g, requires_grad=True)
C_g_t = torch.randn(B_g, N_g, L_g, requires_grad=True)
D_g_skip = torch.randn(D_g, requires_grad=True)

y_g = selective_scan_ref(u_g, delta_g, A_g, B_g_t, C_g_t, D=D_g_skip)
loss = y_g.sum()
loss.backward()

print("Gradient shapes:")
for name, param in [("u", u_g), ("delta", delta_g), ("A", A_g), ("B", B_g_t), ("C", C_g_t), ("D", D_g_skip)]:
    print(f"  d(loss)/d({name}): shape={tuple(param.grad.shape)}, "
          f"norm={param.grad.norm().item():.4f}, finite={torch.isfinite(param.grad).all().item()}")

## The Full Mamba Block

The selective scan is the core of the Mamba block, but the full block adds several more components:

```
hidden_states (B, L, D)
    |
    v
[in_proj] -- Linear(D, 2*D_inner) --> split into x and z
    |                                       |
    v                                       |
[conv1d] -- causal depthwise conv           |
    |                                       |
    v                                       |
[SiLU activation]                           |
    |                                       |
    v                                       |
[x_proj] -- Linear(D_inner, dt_rank+2N)    |
    |                                       |
    v                                       |
  split: dt, B_t, C_t                      |
    |                                       |
    v                                       |
[dt_proj] -- Linear(dt_rank, D_inner)      |
    |                                       |
    v                                       |
[selective_scan](u, delta, A, B_t, C_t, D, z)
    |
    v
[out_proj] -- Linear(D_inner, D)
    |
    v
output (B, L, D)
```

See `src/mamba_minimal/model.py` for the implementation.


In [ ]:
from mamba_minimal.model import MambaBlock

# Create a block with the same config as mamba-130m
block = MambaBlock(d_model=768, d_state=16, d_conv=4, expand=2)

# Count parameters
total_params = sum(p.numel() for p in block.parameters())
print(f"MambaBlock parameters: {total_params:,}")
print(f"  in_proj:  {block.in_proj.weight.shape}")
print(f"  conv1d:   {block.conv1d.weight.shape}")
print(f"  x_proj:   {block.x_proj.weight.shape}")
print(f"  dt_proj:  {block.dt_proj.weight.shape}")
print(f"  A_log:    {block.A_log.shape}")
print(f"  D:        {block.D.shape}")
print(f"  out_proj: {block.out_proj.weight.shape}")

# Forward pass
x = torch.randn(2, 16, 768)
y = block(x)
print(f"\nForward: {tuple(x.shape)} -> {tuple(y.shape)}")

## Official Model Parity

We validate our MambaBlock by loading weights from `state-spaces/mamba-130m-hf` and comparing outputs. The parity script (`scripts/official_parity.py`) loads each mixer layer and checks that our implementation produces identical outputs.

**Result:** max absolute error = 0.0 across layers 0, 5, 11, 17, 23. Our implementation is numerically exact.

**Next notebook:** We explore how to parallelize the sequential scan for better GPU utilization.
